# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassan-tech-pro/ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Provisional lane: Refresh / Content Opportunity Scoring.** I want to help an editor decide which existing content page to review first. The output would be a ranked review queue, not an automatic instruction to rewrite a page. The starter dataset has page-level search, engagement, age, and recent trend signals, so it can support a first scoring exercise. This is a provisional direction: the starter label is only a proxy, and later work must test whether a score helps beyond a simple rule.

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
task_type = "ranking / scoring"
print(f"Provisional ML task: {task_type}")

Provisional ML task: ranking / scoring


## 2. Target or proxy

**Task type: ranking / scoring.** I would rank eligible pages for human review using a priority score.

**Starter proxy:** `decline_proxy = 1` when `trend_direction == "down"`; otherwise it is 0. The source trend bucket is based on the recent 30-day impression change versus the previous 30 days, so this is a rule-derived, same-snapshot proxy, not an independently observed future outcome. I must not use `trend_direction` or `trend_pct` as predictor features because they define the proxy. A later capstone should build a future-window outcome after its feature window.

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
proxy_rule = 'trend_direction == "down"'
print(f"Starter-only decline proxy rule: {proxy_rule}")

Starter-only decline proxy rule: trend_direction == "down"


## 3. Success metric

**Primary metric: Precision@50.** Of the 50 highest-ranked eligible pages, measure the share marked `decline_proxy = 1`. Compare it with the proxy prevalence and a transparent fixed-rule baseline on the same client-grouped holdout split. For a first prototype, a result is worth further study only if Precision@50 is at least 5 percentage points above the proxy prevalence and also beats that rule. This evaluates agreement with the starter proxy, not future refresh success.

*One metric you can defend. What number means 'good'?*

In [3]:
metric_name = "Precision@50"
top_k = 50
minimum_lift_over_proxy_prevalence = 0.05
print(
    f"Primary metric: {metric_name}; provisional minimum lift over proxy prevalence: "
    f"{minimum_lift_over_proxy_prevalence:.0%}"
)

Primary metric: Precision@50; provisional minimum lift over proxy prevalence: 5%


## 4. The unit of analysis, as a real dataframe

**One row = one eligible content page** in the anonymized starter snapshot. Following the repo's preparation pipeline, this starter slice excludes pages with zero impressions and pages younger than 90 days, then keeps one row per `content_id`. The preview omits pseudonymous IDs; they are used only for deduplication. The `decline_proxy` column is created from the defined trend bucket, and is not a feature.

*Load your lane's slice and show it: one row = one what?*

In [4]:
!pip install panda

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
  Created wheel for panda: filename=panda-0.3.1-py3-none-any.whl size=7288 sha256=2690898e500c0a4e9706a061aa33e4fe1e61b066c709626ac8ebfef80b13d481
  Stored in directory: c:\users\abdullah\appdata\local\pip\cache\wheels\48\ef\ba\3f5639b09268e985269fca8e9cd9f056a93258319e60e608d3
Successfully built panda

   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptoo

In [6]:
from pathlib import Path

import pandas as pd

relative_data_path = Path("content_refresh_anonymized.csv")
data_candidates = [
    relative_data_path,
    Path("content_refresh_anonymized.csv"),
    Path("content_refresh_anonymized.csv"),
]
data_path = next((candidate for candidate in data_candidates if candidate.is_file()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv from the current directory."
    )

raw_pages = pd.read_csv(data_path)
required_columns = {
    "content_id",
    "content_type",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "trend_direction",
}
missing_columns = required_columns.difference(raw_pages.columns)
if missing_columns:
    raise ValueError(f"Starter CSV is missing required columns: {sorted(missing_columns)}")

raw_pages["content_age_days"] = pd.to_numeric(raw_pages["content_age_days"], errors="coerce")
raw_pages["impressions_90d"] = pd.to_numeric(raw_pages["impressions_90d"], errors="coerce")
page_slice = raw_pages.loc[
    raw_pages["impressions_90d"].gt(0) & raw_pages["content_age_days"].ge(90)
].drop_duplicates(subset="content_id").copy()
page_slice["decline_proxy"] = (
    page_slice["trend_direction"].fillna("").astype(str).str.lower().eq("down").astype("int8")
)

modeling_frame = page_slice[
    [
        "content_type",
        "content_age_days",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "decline_proxy",
    ]
].copy()
print(f"Rows in raw snapshot: {len(raw_pages):,}")
print(f"Eligible, deduplicated page rows: {len(modeling_frame):,}")
print(f"Starter decline-proxy prevalence: {modeling_frame['decline_proxy'].mean():.1%}")
modeling_frame.head(10)

Rows in raw snapshot: 30,000
Eligible, deduplicated page rows: 30,000
Starter decline-proxy prevalence: 54.2%


,content_type,content_age_days,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,days_since_last_update,decline_proxy
0,keyword article,187,3803,29,17,0.76,10.6,20,1
1,keyword article,445,15320,7,9,0.05,20.3,25,1
2,keyword article,141,12581,11,11,0.09,36.5,20,1
3,keyword article,463,11751,58,78,0.49,6.2,22,0
4,keyword article,263,19140,24,145,0.13,44.0,14,1
5,keyword article,147,3970,1,5,0.03,8.5,20,1
6,keyword article,90,20,0,1,0.00,7.0,20,1
7,keyword article,445,1724,1,28,0.06,21.2,22,0
8,keyword article,90,32574,29,68,0.09,46.0,20,1
9,keyword article,257,1240,2,3,0.16,4.9,104,1


## 5. Why ML beats a fixed rule here

This is a hypothesis to test, not a result: pages with similar impression volume may deserve different review priority depending on age, position, CTR, and update recency together. A fixed rule is the required baseline and may be enough. A learned score earns a place only if, on the same client-grouped holdout, it improves Precision@50 over that rule and the proxy base rate by a useful amount. Editors would use the ranked queue to inspect a page and choose whether to monitor, refresh, or investigate it; a false positive wastes review time, while a false negative can delay attention to a page with weakening visibility. The score supports review, not automatic edits or causal claims.

*What makes the pattern too messy for an if-statement?*

In [7]:
candidate_feature_columns = [
    "content_age_days",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_type",
]
forbidden_feature_columns = {"trend_direction", "trend_pct", "decline_proxy"}
missing_candidate_features = set(candidate_feature_columns).difference(modeling_frame.columns)
assert not missing_candidate_features, f"Missing candidate columns: {sorted(missing_candidate_features)}"
assert not forbidden_feature_columns.intersection(candidate_feature_columns)
print("Candidate signals are present; proxy-defining fields are excluded from the feature list.")
print(f"Candidate signals: {', '.join(candidate_feature_columns)}")

Candidate signals are present; proxy-defining fields are excluded from the feature list.
Candidate signals: content_age_days, impressions_90d, ctr, avg_position, days_since_last_update, content_type


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.